### Parameter Pipeline

 This notebook uses an exhaustive search to evalute parameters with StratifiedKFold, Logistic Regression.  Results were sorted by F2. The results with'CSF Amyloid (pg/mL)', 'CSF Phosphorylated tau (pg/mL)' showed an F2 and Recall of 0.80. Hypertuning using GridSearchCV produced the parameters LogisticRegression(C=10, class_weight=None, penalty='l2', max_iter=1000) which resulted in slightly higher Accuracy, and a stable Recall and F2 of 0.800. The False Negatives showed the most improvement from 8 to 2.
```
Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])
```
|Accuracy|Precision| ROC_UAC | Recall | F2 |False Negatives | False Negative Rate|
|--|--|--|--|--|--|--|
|0.73|0.70|0.78| 0.82 |  0.80 | 0.80 | 0.17|

```
 Pipeline([('sc', StandardScaler()),('clf', LogisticRegression(C=10, class_weight=None, penalty='l2', max_iter=1000))])
```

|Accuracy|Precision| ROC_UAC | Recall | F2| False Negatives | False Negative Rate|
|--|--|--|--|--|--|--|
|0.78|0.80|0.85|0.8000|0.8000| 2 | 0.2|

NOTE: The test size was only 18 samples, so the FNR is based on 10 Yes's that were 2 predicted to be False. This test needs more data to explore these parameters more fully.

# Determine the best parameters


In [1]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
import seaborn as sns
sns.set_palette("pastel")
import plotly.express as px
import matplotlib.pyplot as plt
import math
import numpy as np

pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
%matplotlib inline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
import pandas as pd

# 1. Load data, keep only patients with a known outcome (MCI patients)
df = pd.read_csv('data/plasma_lipidomics.csv')
mci = df[df["Progression to Alzheimer's Disease"].notna()].copy()

# 2. Fill missing numeric values with the column median
numeric_cols = ['Age', 'MMSE', 'CSF Amyloid (pg/mL)', 'CSF Total tau (pg/mL)',
                 'CSF Phosphorylated tau (pg/mL)']
for col in numeric_cols:
    median_val = mci[col].median()
    mci[col] = mci[col].fillna(median_val)

# 3. Fill missing categorical values with the most common value
mode_val = mci['APOE4'].mode()[0]
mci['APOE4'] = mci['APOE4'].fillna(mode_val)

# 4. Convert categorical text columns to numeric (0/1)
mci['Sex'] = (mci['Sex'] == 'Male').astype(int)           # Male=1, Female=0
mci['APOE4'] = (mci['APOE4'] == 'Yes').astype(int)        # carries APOE4 allele=1, no=0

# 5. Set the Target
mci['Target'] = (mci["Progression to Alzheimer's Disease"] == 'Yes').astype(int)

# 5. Final feature set + target
feature_cols = numeric_cols + ['Sex', 'APOE4']
X = mci[feature_cols]
y = mci['Target']

In [3]:
X

,Age,MMSE,CSF Amyloid (pg/mL),CSF Total tau (pg/mL),CSF Phosphorylated tau (pg/mL),Sex,APOE4
64,69,23,595.0,465.0,75.0,1,0
104,70,27,1845.0,353.0,92.4,1,0
105,73,29,928.0,531.0,176.0,0,0
106,68,23,619.0,477.0,142.0,0,1
107,78,27,784.0,1231.0,394.0,1,0
...,...,...,...,...,...,...,...
187,75,25,465.0,67.1,22.4,1,0
188,70,27,965.0,408.0,52.7,1,1
189,77,22,604.0,332.0,63.5,0,0
190,77,22,314.0,692.0,84.4,0,1


In [4]:
from sklearn.metrics import precision_score, accuracy_score, roc_auc_score, confusion_matrix, classification_report, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
import pandas as pd

def evaluate_pipeline(terms, X, y, model, verbose=False):
    # ---- 5-fold cross-validation, honest out-of-fold predictions ----
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    y_pred = cross_val_predict(model, X, y, cv=cv, method='predict')
    y_proba = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]

    # ---- Confusion matrix ----
    cm = confusion_matrix(y, y_pred, labels=[0, 1])

    if verbose:
        display_terms = ' '.join(terms)
        print(f"Predicting 'Progression to Alzheimer's Disease' from {display_terms}\n")
        print(pd.DataFrame(cm, index=['Actual: No', 'Actual: Yes'], columns=['Pred: No', 'Pred: Yes']))
        print(classification_report(y, y_pred, target_names=['No progression', 'Progressed'], digits=3))
        print('Accuracy:', accuracy_score(y, y_pred))
        print('AUC:', roc_auc_score(y, y_proba))

    # ---- Extract metrics ----
    tn, fp, fn, tp = cm.ravel()

    precision = precision_score(y, y_pred)
    recall =  recall_score(y, y_pred, zero_division=0),
    accuracy = accuracy_score(y, y_pred)
    auc = roc_auc_score(y, y_proba)
    fnr = fn / (fn + tp) if (fn + tp) > 0 else float('nan')

    return {
        'features': terms,
        'n_features': len(terms),
        'accuracy': accuracy,
        'precision': precision,
        'recall' : recall,
        'auc': auc,
        'false_negatives': fn,
        'false_negative_rate': fnr,
    }

In [5]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, make_scorer, confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd

def evaluate_performance(terms, X, y, verbose=False):
    # ---- 5-fold cross-validation, honest out-of-fold predictions ----
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    model = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])

    y_pred = cross_val_predict(model, X, y, cv=cv, method='predict')
    y_proba = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]

    # ---- Confusion matrix ----
    cm = confusion_matrix(y, y_pred, labels=[0, 1])

    if verbose:
        display_terms = ' '.join(terms)
        print(f"Predicting 'Progression to Alzheimer's Disease' from {display_terms}\n")
        print(pd.DataFrame(cm, index=['Actual: No', 'Actual: Yes'], columns=['Pred: No', 'Pred: Yes']))
        print(classification_report(y, y_pred, target_names=['No progression', 'Progressed'], digits=3))
        print('Accuracy:', accuracy_score(y, y_pred))
        print('AUC:', roc_auc_score(y, y_proba))

    # ---- Extract metrics ----
    tn, fp, fn, tp = cm.ravel()

    precision = precision_score(y, y_pred)
    accuracy = accuracy_score(y, y_pred)
    roc_auc = roc_auc_score(y, y_proba)
    fnr = fn / (fn + tp) if (fn + tp) > 0 else float('nan')

    return {
        'features': terms,
        'n_features': len(terms),
        'accuracy': accuracy,
        'recall': recall_score(y, y_pred, zero_division=0),
        'f2': fbeta_score(y, y_pred, beta=2, zero_division=0),
        'precision': precision,
        'roc_auc': roc_auc,
        'false_negatives': fn,
        'false_negative_rate': fnr,
    }

In [6]:
import itertools
feature_list = list(X.columns)
all_combos = []
for r in range(1, len(feature_list) + 1):
    all_combos.extend(itertools.combinations(feature_list, r))

results = []
for combo in all_combos:
    cols = list(combo)
    print(cols)
    result = evaluate_performance(cols, X[cols], y)
    results.append(result)

results_df = pd.DataFrame(results)

results_df

['Age']
['MMSE']
['CSF Amyloid (pg/mL)']
['CSF Total tau (pg/mL)']
['CSF Phosphorylated tau (pg/mL)']
['Sex']
['APOE4']
['Age', 'MMSE']
['Age', 'CSF Amyloid (pg/mL)']
['Age', 'CSF Total tau (pg/mL)']
['Age', 'CSF Phosphorylated tau (pg/mL)']
['Age', 'Sex']
['Age', 'APOE4']
['MMSE', 'CSF Amyloid (pg/mL)']
['MMSE', 'CSF Total tau (pg/mL)']
['MMSE', 'CSF Phosphorylated tau (pg/mL)']
['MMSE', 'Sex']
['MMSE', 'APOE4']
['CSF Amyloid (pg/mL)', 'CSF Total tau (pg/mL)']
['CSF Amyloid (pg/mL)', 'CSF Phosphorylated tau (pg/mL)']
['CSF Amyloid (pg/mL)', 'Sex']
['CSF Amyloid (pg/mL)', 'APOE4']
['CSF Total tau (pg/mL)', 'CSF Phosphorylated tau (pg/mL)']
['CSF Total tau (pg/mL)', 'Sex']
['CSF Total tau (pg/mL)', 'APOE4']
['CSF Phosphorylated tau (pg/mL)', 'Sex']
['CSF Phosphorylated tau (pg/mL)', 'APOE4']
['Sex', 'APOE4']
['Age', 'MMSE', 'CSF Amyloid (pg/mL)']
['Age', 'MMSE', 'CSF Total tau (pg/mL)']
['Age', 'MMSE', 'CSF Phosphorylated tau (pg/mL)']
['Age', 'MMSE', 'Sex']
['Age', 'MMSE', 'APOE4']
['A

,features,n_features,accuracy,recall,f2,precision,roc_auc,false_negatives,false_negative_rate
0,[Age],1,0.460674,0.744681,0.675676,0.492958,0.489108,12,0.255319
1,[MMSE],1,0.584270,0.617021,0.614407,0.604167,0.610689,18,0.382979
2,[CSF Amyloid (pg/mL)],1,0.719101,0.829787,0.799180,0.696429,0.751013,8,0.170213
3,[CSF Total tau (pg/mL)],1,0.719101,0.659574,0.679825,0.775000,0.727204,16,0.340426
4,[CSF Phosphorylated tau (pg/mL)],1,0.640449,0.723404,0.705394,0.641509,0.715552,13,0.276596
...,...,...,...,...,...,...,...,...,...
122,"[Age, MMSE, CSF Amyloid (pg/mL), CSF Phosphorylated tau (pg/mL), Sex, APOE4]",6,0.685393,0.702128,0.702128,0.702128,0.771530,14,0.297872
123,"[Age, MMSE, CSF Total tau (pg/mL), CSF Phosphorylated tau (pg/mL), Sex, APOE4]",6,0.685393,0.680851,0.686695,0.711111,0.769504,15,0.319149
124,"[Age, CSF Amyloid (pg/mL), CSF Total tau (pg/mL), CSF Phosphorylated tau (pg/mL), Sex, APOE4]",6,0.764045,0.744681,0.754310,0.795455,0.781662,12,0.255319
125,"[MMSE, CSF Amyloid (pg/mL), CSF Total tau (pg/mL), CSF Phosphorylated tau (pg/mL), Sex, APOE4]",6,0.741573,0.744681,0.747863,0.760870,0.797872,12,0.255319


In [7]:
# ---- Filter: FNR <= 0.20, sort by accuracy desc, AUC as tiebreaker ----
best_df = (
    results_df.sort_values(by=['f2'], ascending=[False]).reset_index(drop=True)
)
print(f"{len(best_df)} of {len(results_df)} combinations meet the FNR <= 0.20 threshold\n")
print(best_df.head(10))

127 of 127 combinations meet the FNR <= 0.20 threshold

                                                                             features  \
0                               [CSF Amyloid (pg/mL), CSF Phosphorylated tau (pg/mL)]   
1                                                          [Age, CSF Amyloid (pg/mL)]   
2                                                               [CSF Amyloid (pg/mL)]   
3                                                          [CSF Amyloid (pg/mL), Sex]   
4                                                     [Age, CSF Amyloid (pg/mL), Sex]   
5                          [CSF Amyloid (pg/mL), CSF Phosphorylated tau (pg/mL), Sex]   
6                          [Age, CSF Amyloid (pg/mL), CSF Phosphorylated tau (pg/mL)]   
7                     [Age, CSF Amyloid (pg/mL), CSF Phosphorylated tau (pg/mL), Sex]   
8                                  [MMSE, CSF Amyloid (pg/mL), CSF Total tau (pg/mL)]   
9  [MMSE, CSF Amyloid (pg/mL), CSF Total tau (pg/mL), 

In [8]:
best = best_df.iloc[0]

In [9]:
best.features

['CSF Amyloid (pg/mL)', 'CSF Phosphorylated tau (pg/mL)']

In [10]:
best.T

features               [CSF Amyloid (pg/mL), CSF Phosphorylated tau (pg/mL)]
n_features                                                                 2
accuracy                                                            0.730337
recall                                                              0.829787
f2                                                                  0.802469
precision                                                           0.709091
roc_auc                                                             0.779382
false_negatives                                                            8
false_negative_rate                                                 0.170213
Name: 0, dtype: object

In [11]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.metrics import make_scorer, fbeta_score, precision_score, accuracy_score, roc_auc_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd

best_features = ['CSF Amyloid (pg/mL)', 'CSF Phosphorylated tau (pg/mL)']
X_best = X[best_features]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])

# Modest grid given the small sample size (89 patients)
param_grid = {
    'clf__C': [0.01, 0.1, 1, 10, 100],
    'clf__penalty': ['l2'],
    'clf__class_weight': [None, 'balanced'],
}

f2_scorer = make_scorer(fbeta_score, beta=2,zero_division=0)

scoring_options = {
    'recall': 'recall',
    'f2': f2_scorer,
}

def run_grid_search(scoring_name, scoring):
    grid = GridSearchCV(pipe, param_grid, scoring=scoring, cv=cv, n_jobs=1)
    grid.fit(X_best, y)

    best_model = grid.best_estimator_

    # Evaluate the winning hyperparameters with honest out-of-fold predictions
    y_pred = cross_val_predict(best_model, X_best, y, cv=cv, method='predict')
    y_proba = cross_val_predict(best_model, X_best, y, cv=cv, method='predict_proba')[:, 1]

    cm = confusion_matrix(y, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    fnr = fn / (fn + tp) if (fn + tp) > 0 else float('nan')

    return {
        'scoring': scoring_name,
        'best_params': grid.best_params_,
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred),
        'roc_auc': roc_auc_score(y, y_proba),
        'recall': recall_score(y, y_pred, zero_division=0),
        'f2': fbeta_score(y, y_pred, beta=2, zero_division=0),
        'false_negatives': int(fn),
        'false_negative_rate': fnr,
    }

results = [run_grid_search(name, scoring) for name, scoring in scoring_options.items()]
comparison_df = pd.DataFrame(results)
print(comparison_df)

  scoring                                                        best_params  \
0  recall  {'clf__C': 0.01, 'clf__class_weight': None, 'clf__penalty': 'l2'}   
1      f2    {'clf__C': 10, 'clf__class_weight': None, 'clf__penalty': 'l2'}   

   accuracy  precision   roc_auc    recall        f2  false_negatives  \
0  0.595506   0.579710  0.777356  0.851064  0.778210                7   
1  0.741573   0.722222  0.776849  0.829787  0.805785                8   

   false_negative_rate  
0             0.148936  
1             0.170213  


In [12]:
comparison_df.iloc[0]

scoring                                                                           recall
best_params            {'clf__C': 0.01, 'clf__class_weight': None, 'clf__penalty': 'l2'}
accuracy                                                                        0.595506
precision                                                                        0.57971
roc_auc                                                                         0.777356
recall                                                                          0.851064
f2                                                                               0.77821
false_negatives                                                                        7
false_negative_rate                                                             0.148936
Name: 0, dtype: object

In [13]:
comparison_df.iloc[1]

scoring                                                                             f2
best_params            {'clf__C': 10, 'clf__class_weight': None, 'clf__penalty': 'l2'}
accuracy                                                                      0.741573
precision                                                                     0.722222
roc_auc                                                                       0.776849
recall                                                                        0.829787
f2                                                                            0.805785
false_negatives                                                                      8
false_negative_rate                                                           0.170213
Name: 1, dtype: object

F2 wins 
'clf', LogisticRegression(C=10, class_weight=None, penalty='l2', max_iter=1000

## Train the model

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, roc_auc_score
)
import pandas as pd

best_features = ['CSF Amyloid (pg/mL)', 'CSF Phosphorylated tau (pg/mL)']
X_best = X[best_features]

# ---- Stratified train/test split (80/20), preserves class balance ----
X_train, X_test, y_train, y_test = train_test_split(
    X_best, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train size: {len(X_train)}  ({y_train.sum()} Yes / {len(y_train) - y_train.sum()} No)")
print(f"Test size:  {len(X_test)}  ({y_test.sum()} Yes / {len(y_test) - y_test.sum()} No)")

# ---- Build and fit the final model ----
final_model = Pipeline([
    ('sc', StandardScaler()),
    ('clf', LogisticRegression(C=10, class_weight=None, penalty='l2', max_iter=1000))
])

final_model.fit(X_train, y_train)

# ---- Evaluate on the held-out test set ----
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print(pd.DataFrame(cm, index=['Actual: No', 'Actual: Yes'], columns=['Pred: No', 'Pred: Yes']))
print(classification_report(y_test, y_pred, target_names=['No progression', 'Progressed'], digits=3))

tn, fp, fn, tp = cm.ravel()
fnr = fn / (fn + tp) if (fn + tp) > 0 else float('nan')

test_result = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_proba),
    'f2': fbeta_score(y_test, y_pred, beta=2, zero_division=0),
    'false_negatives': int(fn),
    'false_negative_rate': fnr,
}
print(test_result)

Train size: 71  (37 Yes / 34 No)
Test size:  18  (10 Yes / 8 No)
             Pred: No  Pred: Yes
Actual: No          6          2
Actual: Yes         2          8
                precision    recall  f1-score   support

No progression      0.750     0.750     0.750         8
    Progressed      0.800     0.800     0.800        10

      accuracy                          0.778        18
     macro avg      0.775     0.775     0.775        18
  weighted avg      0.778     0.778     0.778        18

{'accuracy': 0.7777777777777778, 'precision': 0.8, 'recall': 0.8, 'roc_auc': 0.85, 'f2': 0.8, 'false_negatives': 2, 'false_negative_rate': np.float64(0.2)}


## The model

In [15]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

best_features = ['CSF Amyloid (pg/mL)', 'CSF Phosphorylated tau (pg/mL)']
X_best = X[best_features]

# ---- Build and fit on the FULL dataset (all 89 patients) ----
deployable_model = Pipeline([
    ('sc', StandardScaler()),
    ('clf', LogisticRegression(C=3, class_weight=None, penalty='l2', max_iter=1000))
])

deployable_model.fit(X_best, y)

# ---- Save to disk ----
joblib.dump(deployable_model, 'alzheimers_progression_model.joblib')
print("Model saved to alzheimers_progression_model.joblib")

# ---- Also save the feature list, since the model needs columns in this exact order ----
joblib.dump(best_features, 'alzheimers_progression_model_features.joblib')

Model saved to alzheimers_progression_model.joblib


['alzheimers_progression_model_features.joblib']

# Load the Model

In [16]:
import joblib
import pandas as pd

model = joblib.load('alzheimers_progression_model.joblib')
features = joblib.load('alzheimers_progression_model_features.joblib')

# new_patients must be a DataFrame with the same columns, in the same order
def predict_progression(new_patients_df):
    X_new = new_patients_df[features]
    predictions = model.predict(X_new)
    probabilities = model.predict_proba(X_new)[:, 1]
    return pd.DataFrame({
        'predicted_progression': predictions,
        'progression_probability': probabilities
    }, index=new_patients_df.index)

# example:
# results = predict_progression(new_patients_df)

In [17]:
import pandas as pd

feature_cols = ['CSF Amyloid (pg/mL)', 'CSF Phosphorylated tau (pg/mL)']

# Raw array: [amyloid, p-tau] per patient
test_array = [
    [368.2, 119.2],   # High-risk profile
    [604.0, 63.5],    # Median profile
    [1062.0, 37.1],   # Low-risk profile    
    [1845.0, 22.4],   # Extreme low-risk (observed max amyloid + min p-tau)
]

test_labels = [
    'High-risk profile',
    'Median profile',
    'Low-risk profile',    
    'Extreme low-risk',
]

test_patients = pd.DataFrame(test_array, columns=feature_cols, index=test_labels)
test_patients

,CSF Amyloid (pg/mL),CSF Phosphorylated tau (pg/mL)
High-risk profile,368.2,119.2
Median profile,604.0,63.5
Low-risk profile,1062.0,37.1
Extreme low-risk,1845.0,22.4


In [18]:




model = joblib.load('alzheimers_progression_model.joblib')  # adjust path if needed

predictions = model.predict(test_patients)
probabilities = model.predict_proba(test_patients)[:, 1]

results = test_patients.copy()
results['predicted_progression'] = predictions
results['progression_probability'] = probabilities.round(3)

results

,CSF Amyloid (pg/mL),CSF Phosphorylated tau (pg/mL),predicted_progression,progression_probability
High-risk profile,368.2,119.2,1,0.804
Median profile,604.0,63.5,1,0.564
Low-risk profile,1062.0,37.1,0,0.180
Extreme low-risk,1845.0,22.4,0,0.013
